# Feature Engineering & Data Preparation

This notebook demonstrates how to use Databricks for feature engineering and preparation, with data ready for AzureML model training and scoring.

## Key Steps:
1. Load raw data from Azure Storage
2. Perform feature engineering using Spark
3. Create feature store tables
4. Export features for AzureML consumption
5. Transform data into AzureML-compatible format

In [ ]:
import pandas as pd
import pyspark.sql.functions as F
from pyspark.sql.types import *
from pyspark.sql import Window
from datetime import datetime, timedelta
import json
import logging

# Latest Databricks SDK (databricks-sdk)
try:
    from databricks.sdk import WorkspaceClient
    from databricks.sdk.service.jobs import RunsJobsAPI
    print("✓ Databricks SDK (databricks-sdk) imported successfully")
except ImportError:
    print("Note: databricks-sdk not installed. Install with: pip install databricks-sdk")
    WorkspaceClient = None

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(name)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Configuration
STORAGE_ACCOUNT = dbutils.widgets.get("storage_account", "")
CONTAINER = dbutils.widgets.get("container", "raw-data")
OUTPUT_PATH = f"abfss://{CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/processed"

print(f"Storage Account: {STORAGE_ACCOUNT}")
print(f"Output Path: {OUTPUT_PATH}")
if WorkspaceClient:
    print("✓ Databricks SDK available for workspace operations")

## Step 1: Load and Explore Raw Data

In [ ]:
# Example: Load from parquet
df = spark.read.parquet(f"{OUTPUT_PATH}/raw_data")

# Display schema and sample data
print("Schema:")
df.printSchema()

print("\nSample Data:")
display(df.limit(5))

print(f"\nTotal Records: {df.count()}")

## Step 2: Data Cleaning & Transformation

In [ ]:
# Remove null values
df_clean = df.dropna()

# Handle duplicates
df_clean = df_clean.dropDuplicates()

# Data type conversions
df_clean = df_clean.withColumn("timestamp", F.col("timestamp").cast("timestamp"))

print(f"Records after cleaning: {df_clean.count()}")
display(df_clean.limit(3))

## Step 3: Feature Engineering

In [ ]:
# Create derived features
df_features = df_clean.withColumn(
    "hour", F.hour(F.col("timestamp"))
).withColumn(
    "day_of_week", F.dayofweek(F.col("timestamp"))
).withColumn(
    "is_weekend", F.when((F.col("day_of_week") > 5), 1).otherwise(0)
)

# Aggregation-based features (example: last 7 days average)
window_spec = (
    Window
    .partitionBy("customer_id")
    .orderBy(F.col("timestamp").cast("long"))
    .rangeBetween(-7*86400, 0)  # 7 days in seconds
)

from pyspark.sql.window import Window

df_features = df_features.withColumn(
    "avg_amount_7d", F.avg(F.col("amount")).over(window_spec)
)

print("Features created:")
display(df_features.select("customer_id", "amount", "hour", "day_of_week", "is_weekend", "avg_amount_7d").limit(5))

## Step 4: Create Feature Store Table (Unity Catalog)

In [ ]:
# Configure for Unity Catalog
CATALOG = "main"  # or your catalog name
SCHEMA = "features"
TABLE_NAME = "customer_features"

# Create schema if not exists
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")

# Write features to Unity Catalog table
df_features.write.mode("overwrite").option("mergeSchema", "true").saveAsTable(
    f"{CATALOG}.{SCHEMA}.{TABLE_NAME}"
)

print(f"Feature table created: {CATALOG}.{SCHEMA}.{TABLE_NAME}")

# Verify table
feature_df = spark.table(f"{CATALOG}.{SCHEMA}.{TABLE_NAME}")
display(feature_df.limit(5))

## Step 5: Export for AzureML (Train/Score Format)

In [ ]:
# Select columns relevant for ML model
ml_columns = [
    "customer_id", "amount", "hour", "day_of_week", "is_weekend", 
    "avg_amount_7d", "target_label"  # Ensure target label is included for training
]

df_ml = df_features.select(*ml_columns).na.fill(0)

# Convert to Pandas for easier handling
pdf = df_ml.toPandas()

# Split into train/test
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(pdf, test_size=0.2, random_state=42)

print(f"Train set size: {len(train_df)}")
print(f"Test set size: {len(test_df)}")

# Export to CSV for AzureML
train_path = f"{OUTPUT_PATH}/train_data.csv"
test_path = f"{OUTPUT_PATH}/test_data.csv"

train_df.to_csv(train_path, index=False)
test_df.to_csv(test_path, index=False)

print(f"\nTrain data exported to: {train_path}")
print(f"Test data exported to: {test_path}")

## Step 6: Data Quality Checks

In [ ]:
# Check for missing values
print("Missing Values:")
missing_summary = df_features.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df_features.columns])
display(missing_summary)

# Statistical summary
print("\nStatistical Summary:")
display(df_features.describe())

# Class balance (if classification problem)
print("\nTarget Label Distribution:")
display(df_features.groupBy("target_label").count())